In [1]:
cd /Volumes/Timotej/Dokumenti/Projekti/KG-temporal-relation-extraction

/Volumes/Timotej/Dokumenti/Projekti/KG-temporal-relation-extraction


/Users/timotej/Documents/Projekti/Python/gold-interpreter/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [1]:
cd /workspace/llm-graph-construction

/workspace/llm-graph-construction


/opt/conda/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


# Analyze the results of the full pipeline experiment

In [2]:
import json
from collections import defaultdict, deque
from torch import nn
import torch

In [3]:
gold_graphs = []
pred_graphs = []
texts = []
document_ids = []

main_file = 'pipeline_predictions_bert_bimodal_transitive.txt'


with open(main_file, 'r') as file:
    lines = file.readlines()
    i = 0
    while i < len(lines):
        line = lines[i]
        i+=1
        # if len(line.strip()) == 0:
        #     gold_graphs = []
        #     pred_graphs = []
        #     continue

        while len(line.split("|")) < 4:
            line = line + "\n"
            line += lines[i]
            i+=1
        # print(line)
        pred, gold, document, document_id = line.split('|')
        pred = json.loads(pred)
        gold = json.loads(gold)

        gold_graphs.append(gold)
        pred_graphs.append(pred)
        texts.append(document)
        document_ids.append(document_id)

In [156]:
gold_graphs = []
pred_graphs = []

main_file = 'pipeline_predictions_flert.txt'
gold_relations_replacement_file = 'pipeline_predictions_flert_gold_pairs.txt'
replacement_gold_pairs_relations = []
replacement_gold = []
with open(gold_relations_replacement_file, 'r') as file:
    for line in file:
        line = line.strip()
        if len(line) == 0:
            replacement_gold_pairs_relations = []
        pred, gold = [json.loads(x) for x in line.split('|')]
        replacement_gold_pairs_relations.append(pred)
        replacement_gold.append(gold)

with open(main_file, 'r') as file:
    for i, line in enumerate(file):
        line = line.strip()
        if len(line) == 0:
            gold_graphs = []
            pred_graphs = []
            continue
        # print(line)
        pred, gold = [json.loads(x) for x in line.split('|')]

        replacement_relations = None
        for j in range(len(replacement_gold_pairs_relations)):
            if replacement_gold[j] == gold:
                replacement_relations = replacement_gold_pairs_relations[j]

        if replacement_relations is not None:
            pred = [p for p in pred if p[4] == 0]
            pred = replacement_relations + pred

        print(replacement_relations is not None)
        gold_graphs.append(gold)
        pred_graphs.append(pred)

True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True


In [4]:
pred_graphs[0]

[[[217, 241, 'congestive heart failure'],
  'OVERLAP',
  [247, 267, 'an ejection fraction'],
  1,
  1,
  [[-0.52041095495224, -5.6500020027160645, 2.622945547103882],
   [-10.03493595123291, 11.209965705871582, 3.2235381603240967],
   [0.5674712061882019, -7.840407848358154, 3.086812973022461],
   [-7.460276126861572, 5.014240741729736, 4.2672119140625],
   [-0.2882516384124756, -5.106566905975342, 2.3642640113830566],
   [-2.807512044906616, -2.0571377277374268, 3.1159770488739014],
   [-2.1413986682891846, -2.3665006160736084, 3.0003976821899414],
   [-4.378576278686523, -2.035856246948242, 4.536118984222412],
   [0.6909416913986206, -5.085206508636475, 1.6713122129440308],
   [-6.689397811889648, -0.9925642013549805, 5.67750358581543],
   [-2.3577229976654053, 1.023032784461975, 1.5483850240707397],
   [-4.600739479064941, 0.7797932028770447, 3.530203104019165],
   [-0.0482051819562912, -2.3936874866485596, 1.3849236965179443],
   [4.038782119750977, -7.326831817626953, 0.2717126607

In [4]:
m = nn.Softmax(dim=1)
k = 0
for i in range(len(pred_graphs)):
    k = 0
    for j in range(len(pred_graphs[i])):
        if type(pred_graphs[i][j][5]) is list and type(pred_graphs[i][j][5][0]) is list:
            if k >= len(pred_graphs[i][j][5]):
                k = 0
            # compute softmax of confidence
            pred_graphs[i][j][5] = m(torch.tensor([pred_graphs[i][j][5][k]])).tolist()[0]
            k+=1
        else:
            pred_graphs[i][j][5] = m(torch.tensor([pred_graphs[i][j][5]])).tolist()[0]


In [5]:
pred_graphs = [[(tuple(e1), r, tuple(e2), predicted, gold, confidence) for e1, r, e2, predicted, gold, confidence in graph] for graph in pred_graphs]
gold_graphs = [[(tuple(e1), r, tuple(e2)) for e1, r, e2 in graph] for graph in gold_graphs]

In [6]:
pred_graphs[0]


[((217, 241, 'congestive heart failure'),
  'OVERLAP',
  (247, 267, 'an ejection fraction'),
  1,
  1,
  [0.041343726217746735, 0.00024471277720294893, 0.9584115147590637]),
 ((2287, 2299, "'s admission"),
  'AFTER',
  (29, 38, 'Discharge'),
  0,
  1,
  [5.933470381691563e-10, 0.9996600151062012, 0.00033993119723163545]),
 ((2797, 2805, 'evaluate'),
  'OVERLAP',
  (2643, 2668, 'a cardiac catheterization'),
  1,
  1,
  [0.07451209425926208, 1.6623827832518145e-05, 0.925471305847168]),
 ((2878, 2901, 'ischemic cardiomyopathy'),
  'AFTER',
  (2797, 2805, 'evaluate'),
  1,
  1,
  [2.59390890278155e-06, 0.6785292029380798, 0.32146820425987244]),
 ((2907, 2936, 'Abnormal liver function tests'),
  'OVERLAP',
  (2219, 2243, 'Congestive heart failure'),
  0,
  1,
  [0.06579912453889847, 0.0005316825117915869, 0.9336692094802856]),
 ((2955, 2998, 'an isolated , elevated alkaline phosphatase'),
  'OVERLAP',
  (2907, 2936, 'Abnormal liver function tests'),
  1,
  1,
  [0.0026537084486335516, 0.005

In [6]:
transitivity = {
    ("OVERLAP", "OVERLAP"): "OVERLAP",
    ("BEFORE", "OVERLAP"): "BEFORE",
    ("BEFORE", "BEFORE"): "BEFORE",
    ("OVERLAP", "BEFORE"): "BEFORE",
    ("AFTER", "OVERLAP"): "AFTER",
    ("OVERLAP", "AFTER"): "AFTER",
    ("AFTER", "AFTER"): "AFTER",
    ("AFTER", "BEFORE"): "OVERLAP",
    ("BEFORE", "AFTER"): "OVERLAP"
}

inverse = {
    "BEFORE": "AFTER",
    "AFTER": "BEFORE",
    "OVERLAP": "OVERLAP"
}

In [7]:
def event_overlap(event1, event2):
    e1s = event1[0]
    e1e = event1[1]
    e2s = event2[0]
    e2e = event2[1]
    if e1s >= e2s and e1s < e2e:
        return True
    if e1e > e2s and e1e <= e2e:
        return True
    if e2s > e1s and e2s < e1e:
        return True
    return False

In [8]:
def find_relation(relations, event1, event2):
    graph = defaultdict(dict)
    for r in relations:
        e1 = r[0]
        rel = r[1]
        e2 = r[2]
        graph[e1][e2] = rel
        graph[e2][e1] = inverse[rel]

    def bfs(start, target):
        queue = deque([(start, "OVERLAP")])
        visited = set()

        i = 0
        while queue:
            if i > 1000000:
                return None
            i+=1
            current, accumulated_rel = queue.popleft()
            if current == target:
                return accumulated_rel

            visited.add(current)

            for neighbor, rel in graph[current].items():
                if neighbor not in visited:
                    combined_rel = transitivity.get((accumulated_rel, rel), None)
                    if combined_rel:
                        queue.append((neighbor, combined_rel))

        return None

    return bfs(event1, event2)


In [9]:
def get_relation_from_graph(relation, graph):
    events = set()
    for r in graph:
        events.add(r[0])
        events.add(r[2])
    # find matching events from the graph
    event1 = None
    event2 = None
    for e in events:
        if event_overlap(e, relation[0]):
            event1 = e
        if event_overlap(e, relation[2]):
            event2 = e
    if event1 is None or event2 is None:
        return None
    return find_relation(graph, event1, event2)

In [10]:
scenario = 2

if scenario == 1:
    use_gold_event_pairs = True
    use_relation_detector = False
    do_not_use_non_gold_event_pairs = True
    ignore_relations_that_are_not_in_gold = True

if scenario == 2:
    use_gold_event_pairs = False
    use_relation_detector = False
    do_not_use_non_gold_event_pairs = False
    ignore_relations_that_are_not_in_gold = False

if scenario == 3:
    use_gold_event_pairs = True
    use_relation_detector = False
    do_not_use_non_gold_event_pairs = False
    ignore_relations_that_are_not_in_gold = True

confidence_threshold = 0.1
use_conficence_threshold = True

In [12]:

classes = ["BEFORE", "AFTER", "OVERLAP", "NONE"]
def compare_graphs(gold_graph, pred_graph, confusion_matrix):
    filtered_pred_graph = []
    for rel in pred_graph:
        predicted = rel[3]
        from_gold_pair = rel[4]
        confidence = rel[5]
        if use_conficence_threshold and max(confidence) < confidence_threshold:
            continue
        if use_relation_detector and predicted == 0:
            continue
        if not use_gold_event_pairs and from_gold_pair == 1:
            continue
        if do_not_use_non_gold_event_pairs and from_gold_pair == 0:
            continue
        filtered_pred_graph.append(rel)

    for rel in filtered_pred_graph:
        gold_relation = get_relation_from_graph(rel, gold_graph)
        if gold_relation is None and ignore_relations_that_are_not_in_gold:
            continue

        if gold_relation is None:
            gold_relation_index = len(classes) - 1
        else:
            gold_relation_index = classes.index(gold_relation)

        pred_relation_index = classes.index(rel[1])
        confusion_matrix[pred_relation_index][gold_relation_index] += 1
    for rel in gold_graph:
        if get_relation_from_graph(rel, filtered_pred_graph) is None:
            gold_relation_index = classes.index(rel[1])
            confusion_matrix[len(classes) - 1][gold_relation_index] += 1

In [1]:
relation_detection_model = torch.load("relation-detection-model2.pt")
def predict_relation_presence(pred_graph, document_id):
    dataset = DFDataset()
    dataset.load("pipeline_tmp_bert_all/" + document_id.strip() + ".pt")
    for graph in dataset.generated:
        prediction = relation_detection_model(graph)
        pass
    return pred_graph

NameError: name 'torch' is not defined

In [13]:
confusion_matrix = [[0 for _ in range(len(classes))] for _ in range(len(classes))]
for document_ind in range(len(gold_graphs)):
    pred_graphs[document_ind] = predict_relation_presence(pred_graphs[document_ind], document_ids[document_ind])
    compare_graphs(gold_graphs[document_ind], pred_graphs[document_ind], confusion_matrix)
    print(document_ind)

for row in confusion_matrix:
    print(row)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
[1243, 176, 736, 1928]
[86, 782, 391, 823]
[1330, 1237, 5084, 4881]
[198, 131, 455, 0]


In [24]:
import os, sys

class HiddenPrints:
    def __enter__(self):
        self._original_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')

    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout.close()
        sys.stdout = self._original_stdout

In [26]:
results = []

with HiddenPrints():
    for c in range(85, 100, 1):
        confidence_threshold = c / 100.0
        confusion_matrix = [[0 for _ in range(len(classes))] for _ in range(len(classes))]
        for document_ind in range(len(gold_graphs)):
            compare_graphs(gold_graphs[document_ind], pred_graphs[document_ind], confusion_matrix)
            print(document_ind)
        p,r,f1 = compute_scores(confusion_matrix)
        results.append((confidence_threshold, p, r, f1))
for r in results:
    print(r)

/var/folders/jr/s5vp8jnj0hz66ggccq8bfpw40000gn/T/ipykernel_88822/23691673.py:17: RuntimeWarning: invalid value encountered in divide
  f1 = np.nan_to_num(2 * p * r / (p + r))


(0.85, 0.41936457625839557, 0.6283355947535052, 0.503009730708305)
(0.86, 0.42093934968099006, 0.6281977744636916, 0.5040964742704593)
(0.87, 0.42331817469029326, 0.6280097708502966, 0.5057374361856587)
(0.88, 0.42424, 0.6235888993414863, 0.5049514378213673)
(0.89, 0.425530172766724, 0.6194278903456496, 0.504489637431442)
(0.9, 0.42718283894754483, 0.6174155262201768, 0.5049774652072706)
(0.91, 0.4302375809935205, 0.6136027599802859, 0.50581483926667)
(0.92, 0.4335776863937064, 0.6096794468887492, 0.5067655817355415)
(0.93, 0.4347987386384715, 0.6037347070186735, 0.5055265002426268)
(0.94, 0.4366292569659443, 0.5942849618119568, 0.5034021193530397)
(0.95, 0.44159427695452225, 0.5824235072112145, 0.5023250406882122)
(0.96, 0.44460289600702063, 0.5615906886517944, 0.4962958427723015)
(0.97, 0.45132956322945494, 0.5315289783193992, 0.48815720978656946)
(0.98, 0.46191202997086167, 0.4819748081656291, 0.47173019696755)
(0.99, 0.481083019544558, 0.40682335102350264, 0.44084784751889583)


In [16]:
import numpy as np
def compute_scores(confusion_matrix):
    cm = np.array(confusion_matrix)
    true_pos = np.diag(cm)
    false_pos = np.sum(cm, axis=1) - true_pos
    false_neg = np.sum(cm, axis=0) - true_pos
    print("confusion matrix")
    for row in confusion_matrix:
        print(row)
    print("results")
    print(true_pos)
    print(false_pos)
    print(false_neg)

    p = np.nan_to_num(true_pos / (true_pos+false_pos))
    r = np.nan_to_num(true_pos / (true_pos+false_neg))
    f1 = np.nan_to_num(2 * p * r / (p + r))
    print("p:", p)
    print("r:", r)
    print("f1:", f1)
    ma_p = np.sum(p * (true_pos+false_neg)) / np.sum((true_pos+false_neg))
    ma_r = np.sum(r * (true_pos+false_neg)) / np.sum((true_pos+false_neg))
    ma_f1 = np.sum(f1 * (true_pos+false_neg)) / np.sum((true_pos+false_neg))
    print("precision", ma_p)
    print("recall", ma_r)
    print("f1", ma_f1)

    print()
    print("Temporal awareness metrics:")
    p = np.sum(true_pos) / (np.sum(true_pos[:3]) + np.sum(false_pos[:3]))
    r = np.sum(true_pos) / (np.sum(true_pos[:3]) + np.sum(false_neg[:3]))
    f1 = np.nan_to_num(2 * p * r / (p + r))
    print("p:", p)
    print("r:", r)
    print("f1:", f1)
    return p, r, f1
compute_scores(confusion_matrix)

confusion matrix
[1374, 439, 957, 0]
[453, 849, 577, 0]
[1752, 1603, 7625, 0]
[181, 107, 423, 0]
results
[1374  849 7625    0]
[1396 1030 3355  711]
[2386 2149 1957    0]
p: [0.49602888 0.45183608 0.69444444 0.        ]
r: [0.36542553 0.28318879 0.79576289 0.        ]
f1: [0.42082695 0.34816486 0.74165937 0.        ]
precision 0.6042741637470646
recall 0.6026927784577724
f1 0.5956357207670612

Temporal awareness metrics:
p: 0.6301106916629343
r: 0.6026927784577724
f1: 0.6160968438174481


/var/folders/jr/s5vp8jnj0hz66ggccq8bfpw40000gn/T/ipykernel_88822/23691673.py:16: RuntimeWarning: invalid value encountered in divide
  r = np.nan_to_num(true_pos / (true_pos+false_neg))
/var/folders/jr/s5vp8jnj0hz66ggccq8bfpw40000gn/T/ipykernel_88822/23691673.py:17: RuntimeWarning: invalid value encountered in divide
  f1 = np.nan_to_num(2 * p * r / (p + r))


(0.6301106916629343, 0.6026927784577724, 0.6160968438174481)

In [154]:
graph = pred_graphs[0]
pred_events = set()
for rel in graph:
    pred_events.add(rel[0])
    pred_events.add(rel[2])
graph = gold_graphs[0]
gold_events = set()
for rel in graph:
    gold_events.add(rel[0])
    gold_events.add(rel[2])

c = 0
n = 0
for e in gold_events:
    n += 1
    for e2 in pred_events:
        if event_overlap(e, e2):
            c += 1
print(c/n)

0.9369369369369369


In [155]:
gold_graphs[0]

[((3329, 3343, 'Hypothyroidism'),
  'OVERLAP',
  (3389, 3415, 'her chronic hypothyroidism')),
 ((406, 415, 'albuterol'), 'OVERLAP', (1979, 1999, 'albuterol nebulizers')),
 ((2261, 2273, 'the hospital'), 'OVERLAP', (2194, 2206, 'the hospital')),
 ((2338, 2357, 'gradual improvement'),
  'OVERLAP',
  (1912, 1931, 'the hospitalization')),
 ((2338, 2357, 'gradual improvement'),
  'OVERLAP',
  (2361, 2376, 'her oxygenation')),
 ((2379, 2391, 'Pain control'), 'OVERLAP', (2523, 2532, 'oxycodone')),
 ((2508, 2517, 'MS Contin'), 'OVERLAP', (2523, 2532, 'oxycodone')),
 ((2537, 2554, 'breakthrough pain'),
  'OVERLAP',
  (2410, 2443, 'fairly significant abdominal pain')),
 ((1489, 1499, 'extubation'), 'OVERLAP', (2060, 2070, 'extubation')),
 ((2410, 2443, 'fairly significant abdominal pain'),
  'BEFORE',
  (2508, 2517, 'MS Contin')),
 ((2562, 2568, 'helped'), 'OVERLAP', (2508, 2517, 'MS Contin')),
 ((2615, 2634, 'difficulty coughing'), 'OVERLAP', (2642, 2650, 'the pain')),
 ((2642, 2650, 'the pain'

# Create a training dataset for relation detection module

In [19]:
import pandas as pd
from custom_datasets.dataframe_dataset import DFDataset

def combine_datasets(original_dataset, new_dataset):
    original_dataset.df = pd.concat([original_dataset.df, new_dataset.df])
    original_dataset.generated = original_dataset.generated + new_dataset.generated
    return original_dataset

combined_dataset_train = None
combined_dataset_test = None
for i in range(len(document_ids)):
    document_id = document_ids[i]
    dataset = DFDataset()

    dataset.load("pipeline_tmp_bert_all/" + document_id.strip() + ".pt")
    for graph in dataset.generated:
        event1 = graph["event1_oroginal_position"]
        event2 = graph["event2_oroginal_position"]
        gold_relation = get_relation_from_graph((event1, "", event2), gold_graphs[i])
        y = 0
        if gold_relation is not None:
            y = 1
        graph.y = torch.tensor([y])
    
    if i < len(document_ids) / 2:
        if combined_dataset_train is None:
            combined_dataset_train = dataset
        else:
            combined_dataset_train = combine_datasets(combined_dataset_train, dataset)
    else:
        if combined_dataset_test is None:
            combined_dataset_test = dataset
        else:
            combined_dataset_test = combine_datasets(combined_dataset_test, dataset)

KeyboardInterrupt: 

In [ ]:
combined_dataset_train.save("pipeline_tmp/detection_dataset_train.pt")
combined_dataset_test.save("pipeline_tmp/detection_dataset_test.pt")

# Testing with a timeline

In [16]:
from collections import defaultdict, deque

def sort_events(events, constraints):
    # Create adjacency list and in-degree counter for topological sorting
    graph = defaultdict(set)
    event_to_group = {}
    overlap_groups = []
    
    # Process overlap constraints first to form groups
    for event1, relation, event2 in constraints:
        if relation == "OVERLAP":
            found_group = None
            for group in overlap_groups:
                if event1 in group or event2 in group:
                    group.update([event1, event2])
                    found_group = group
                    break
            if found_group is None:
                new_group = set([event1, event2])
                overlap_groups.append(new_group)
    
    # Assign events to groups and create representative mappings
    group_representative = {}
    for group in overlap_groups:
        rep = next(iter(group))  # Choose an arbitrary representative
        for event in group:
            event_to_group[event] = group
            group_representative[event] = rep
    
    # Create a new event list treating groups as single entities
    unique_events = set(events)
    for group in overlap_groups:
        representative = next(iter(group))
        for event in group:
            if event != representative:
                unique_events.discard(event)
    
    # Ensure all events and groups have in-degree initialized
    in_degree = {event: 0 for event in unique_events}
    
    # Rebuild graph with groups as single units
    for event1, relation, event2 in constraints:
        rep1 = group_representative.get(event1, event1)
        rep2 = group_representative.get(event2, event2)
        
        if relation == "BEFORE" and rep1 != rep2:
            graph[rep1].add(rep2)
            in_degree[rep2] += 1
        elif relation == "AFTER" and rep1 != rep2:
            graph[rep2].add(rep1)
            in_degree[rep1] += 1
    
    # Topological sorting using Kahn's algorithm
    queue = deque([event for event in unique_events if in_degree[event] == 0])
    sorted_events = []
    
    while queue:
        event = queue.popleft()
        sorted_events.append(event)
        for neighbor in graph[event]:
            in_degree[neighbor] -= 1
            if in_degree[neighbor] == 0:
                queue.append(neighbor)
    
    # If not all events are sorted, there's a cycle
    if len(sorted_events) != len(unique_events):
        print(len(sorted_events))
        print(in_degree)
        raise ValueError("Cyclic dependency detected in event constraints")
    
    # Replace representatives with full groups in the final order
    final_order = []
    seen = set()
    for event in sorted_events:
        rep = group_representative.get(event, event)
        if rep in seen:
            continue
        if rep in event_to_group:
            final_order.append(sorted(event_to_group[rep]))
            seen.update(event_to_group[rep])
        else:
            final_order.append(rep)
            seen.add(rep)
    
    return final_order


In [17]:
def convert_graph_to_timeline(graph):
    events = set()
    for relation in graph:
        events.add(tuple(relation[0]))
        events.add(tuple(relation[2]))
    graph = [(tuple(e1), r, tuple(e2)) for e1, r, e2 in graph]
    timeline = sort_events(sorted(list(events), key=lambda e: e[0]), graph)
    return timeline, events

In [18]:
def get_relation_from_timeline(relation, timeline):
    event1_index = -1
    event2_index = -1
    for event_time_index in range(len(timeline)):
        event = timeline[event_time_index]
        if isinstance(event, list):
            for subevent in event:
                if event_overlap(relation[0], subevent):
                    event1_index = event_time_index
                if event_overlap(relation[2], subevent):
                    event2_index = event_time_index
        else:
            if event_overlap(relation[0], event):
                event1_index = event_time_index
            if event_overlap(relation[2], event):
                event2_index = event_time_index
    print(event1_index, event2_index)
    if event1_index == -1 or event2_index == -1:
        return None
    if event1_index == event2_index:
        return "OVERLAP"
    if event1_index < event2_index:
        return "BEFORE"
    if event1_index > event2_index:
        return "AFTER"

In [19]:
classes = ["BEFORE", "AFTER", "OVERLAP", "NONE"]
def compare_graphs_with_timeline(gold_graph, pred_graph, confusion_matrix):
    gold_timeline, gold_events = convert_graph_to_timeline(gold_graph)
    print(gold_timeline)
    for rel in pred_graph:
        gold_relation = get_relation_from_timeline(rel, gold_timeline)
        if gold_relation is None:
            gold_relation_index = len(classes) - 1
        else:
            gold_relation_index = classes.index(gold_relation)
        pred_relation_index = classes.index(rel[1])
        confusion_matrix[pred_relation_index][gold_relation_index] += 1
    # for rel in gold_graph:
    #     if get_relation_from_graph(rel, transitive_closure_generator(pred_graph)) is None:
    #         gold_relation_index = classes.index(rel[1])
    #         confusion_matrix[len(classes) - 1][gold_relation_index] += 1

confusion_matrix = [[0 for _ in range(len(classes))] for _ in range(len(classes))]
for gold, pred in zip(gold_graphs, pred_graphs):
    compare_graphs_with_timeline(gold, pred, confusion_matrix)
    print(confusion_matrix)

32
{(1885, 1888, 'OOB'): 0, (2000, 2014, 'inpatient CMED'): 0, (1673, 1678, 'drops'): 0, (941, 947, 'benzos'): 0, (1283, 1301, 'the medicine floor'): 0, (533, 538, 'alert'): 0, (978, 996, 'seen by cardiology'): 0, (493, 503, 'paroxitine'): 0, (1425, 1435, 'bipolar DO'): 0, (1126, 1153, 'a calcium and glucagon drip'): 1, (1231, 1240, 'extubated'): 0, (1854, 1873, 'suicide precautions'): 0, (1271, 1279, 'transfer'): 0, (1187, 1207, 'aspiration pneumonia'): 0, (1348, 1368, 'involved in his care'): 0, (156, 167, 'hepatitis C'): 0, (1689, 1701, 'glucagon gtt'): 0, (1907, 1917, 'house diet'): 0, (1670, 1672, 'HR'): 0, (750, 767, 'calcium gluconate'): 0, (706, 713, 'a pulse'): 0, (1845, 1851, 'stable'): 0, (901, 918, 'serum tox screens'): 0, (1714, 1728, 'BZD withdrawal'): 0, (235, 240, 'admit'): 0, (636, 645, 'intubated'): 0, (192, 208, 'suicide attempts'): 0, (566, 578, 'unresponsive'): 0, (1613, 1623, 'doing well'): 0, (29, 38, 'Discharge'): 0, (1, 10, 'Admission'): 0, (1074, 1083, 'intuba

ValueError: Cyclic dependency detected in event constraints

In [ ]:
import numpy as np
cm = np.array(confusion_matrix)
true_pos = np.diag(cm)
false_pos = np.sum(cm, axis=0) - true_pos
false_neg = np.sum(cm, axis=1) - true_pos

precision = np.sum(true_pos / (true_pos + false_pos))
recall = np.sum(true_pos / (true_pos + false_neg))

# Evaluation using the older methods

In [44]:
def overlap(event1, event2):
    start1, end1, text1 = event1
    start2, end2, text2 = event2
    if start1 >= start2 and start1 <= end2:
        # print(event1, event2)
        return True
    if end1 >= start2 and end1 <= end2:
        # print(event1, event2)
        return True
    if start2 >= start1 and start2 <= end1:
        # print(event1, event2)
        return True
    return False

def compute_metrics_corrected(gold_graph, predicted_graph):
    gold_graph_index = 0
    correct_predictions = 0
    for predicted in predicted_graph:
        while not overlap(gold_graph[gold_graph_index][0], predicted[0]) or not overlap(gold_graph[gold_graph_index][2], predicted[2]):
            gold_graph_index += 1
        # print(predicted, gold_graph[gold_graph_index])
        correct = predicted[1] == gold_graph[gold_graph_index][1]
        if correct:
            correct_predictions += 1
    precision = correct_predictions / len(predicted_graph)
    recall = correct_predictions / len(gold_graph)
    f1 = 2*(precision*recall)/(precision+recall)
    # print(f1)
    print(len(predicted_graph) / len(gold_graph))
    return correct_predictions, len(predicted_graph), len(gold_graph), precision, recall, f1

In [21]:
pred, gold = documents[0]
for p, g in zip(pred, gold):
    print(p[0][2], "-", p[2][2], "|", g[0][2], "-", g[2][2])

NameError: name 'documents' is not defined

In [45]:
# prepare documents
documents = []
for pred_graph, gold_graph in zip(pred_graphs, gold_graphs):
    filtered_pred_graph = []
    for rel in pred_graph:
        predicted = rel[3]
        from_gold_pair = rel[4]
        if use_relation_detector and predicted == 0:
            continue
        if not use_gold_event_pairs and from_gold_pair == 1:
            continue
        if do_not_use_non_gold_event_pairs and from_gold_pair == 0:
            continue
        filtered_pred_graph.append(rel)
    documents.append((gold_graph, filtered_pred_graph))

In [46]:
system_gold_plus_all = 0
system_plus_gold_all = 0
our_len_all = 0
gold_len_all = 0

macro_p = 0
macro_r = 0
macro_f1 = 0
for gold_graph, predicted_graph in documents:
    correct_predictions, len_pred, len_gold, p, r, f1 = compute_metrics_corrected(gold_graph, predicted_graph)
    print(p, r, f1)
    system_gold_plus_all += correct_predictions
    system_plus_gold_all += correct_predictions
    our_len_all += len_pred
    gold_len_all += len_gold

    macro_p += p
    macro_r += r
    macro_f1 += f1

precision = system_gold_plus_all / our_len_all
recall = system_plus_gold_all / gold_len_all
f1 = 2*(precision*recall)/(precision+recall)
print("micro P:", precision)
print("micro R:", recall)
print("micro F1:", f1)

print("macro P:", macro_p / len(documents))
print("macro R:", macro_r / len(documents))
print("macro F1:", macro_f1 / len(documents))

0.6635514018691588
0.7887323943661971 0.5233644859813084 0.6292134831460674
0.868421052631579
0.803030303030303 0.6973684210526315 0.7464788732394365
0.8247422680412371
0.675 0.5567010309278351 0.6101694915254238
0.7886178861788617
0.711340206185567 0.5609756097560976 0.6272727272727272
0.9310344827586207
0.7777777777777778 0.7241379310344828 0.75
0.9328358208955224
0.64 0.5970149253731343 0.6177606177606176
0.75
0.6666666666666666 0.5 0.5714285714285715
0.8620689655172413
0.64 0.5517241379310345 0.5925925925925927
0.7627118644067796
0.7111111111111111 0.5423728813559322 0.6153846153846154
0.9074074074074074
0.7346938775510204 0.6666666666666666 0.6990291262135923
0.94
0.7446808510638298 0.7 0.7216494845360825
0.6710526315789473
0.7254901960784313 0.4868421052631579 0.5826771653543308
0.8823529411764706
0.5666666666666667 0.5 0.53125
0.9094827586206896
0.6540284360189573 0.5948275862068966 0.6230248306997742
0.7037037037037037
0.6578947368421053 0.46296296296296297 0.5434782608695652
0